# Semantic Document Clustering and Search

This notebook demonstrates sentence embeddings, cosine-similarity search, and KMeans clustering on a small curated corpus. The categories are used for evaluation only; they are not passed into the embedding or clustering model.

## Setup

From the repository root, install the package with `python -m pip install -e .`. The first model load downloads the pretrained sentence encoder.

In [ ]:
from semantic_document_clustering.engine import SemanticDocumentEngine
from semantic_document_clustering.evaluation import evaluate_clustering, evaluate_retrieval
from semantic_document_clustering.io import load_documents, load_queries

documents = load_documents('data/sample_documents.csv')
queries = load_queries('data/search_queries.csv')
engine = SemanticDocumentEngine()
embeddings = engine.encode_documents(documents)
print(f'Encoded {len(documents)} documents into vectors with {embeddings.shape[1]} dimensions.')

## Search by meaning

The sentence encoder represents the query and documents as dense vectors. Cosine similarity ranks documents by vector direction; the score is not a probability.

In [ ]:
query = 'How can a system find useful passages before answering a question?'
hits = engine.search(query, documents, embeddings, top_k=5)
for hit in hits:
    print(f'{hit.score:.3f}  {hit.title}  [{hit.category}]')

## Cluster and evaluate

The example supplies five clusters because the curated corpus has five reference categories. Labels are used only to calculate evaluation scores after clustering. For unlabeled data, use the CLI's silhouette-based k selection as a starting heuristic.

In [ ]:
cluster_ids, chosen_k, silhouette = engine.cluster(documents, embeddings, n_clusters=5)
print(f'Clusters: {chosen_k}; cosine silhouette: {silhouette:.3f}')

retrieval = evaluate_retrieval(engine, documents, queries, embeddings, k=5)
clustering = evaluate_clustering(engine, documents, embeddings)
print({key: value for key, value in retrieval.items() if key != 'per_query'})
print(clustering)

## Interpretation limits

This small, hand-curated corpus is for demonstrating the workflow, not benchmarking semantic search. Review per-query results and try your own corpus before drawing conclusions.